In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
torch.manual_seed(42)
torch.set_printoptions(precision=2,sci_mode=True)


In [ ]:
def olcekli_dikkat(Q,K,V ,maske= None):
  #Tüm anahtar vektör değerlerini alabilmek için son endeksden başlıyoruz
  d_k =Q.size(-1)
  # her sorgu ile her anahtarın benzerliği ölçmek için √d_k ile gösteririz
  skorlar = Q @ K.transpose(-2,-1)/math.sqrt(d_k)
  """
  Maskeli değerler var mı yok mu onu kontrol etmek eğer maskeli alanlar varsa hesabı karıştırmasın diye
  sonsuz bırakıp sonrasında -sonsuzu algoritmada softmax ile olur

  """
  if maske is not None:
    skorlar.masked_fill(maske==0,float("-inf"))
  # skorları ağırlığa çevirmek için
  agirliklar = F.softmax(skorlar,dim=-1)
  cikti = agirliklar@V
  return cikti, agirliklar



In [ ]:
# her satır bir kelimenin vektörü
x = torch.tensor([[1.0,0.0,1.0,0.0],
                   [0.0,2.0,0.0,2.0],
                   [1.0,1.0,1.0,1.0]])
cikti, agirliklar = olcekli_dikkat(x,x,x)
print(cikti)
print(agirliklar)

tensor([[8.45e-01, 7.33e-01, 8.45e-01, 7.33e-01],
        [1.33e-01, 1.85e+00, 1.33e-01, 1.85e+00],
        [5.78e-01, 1.27e+00, 5.78e-01, 1.27e+00]])
tensor([[4.22e-01, 1.55e-01, 4.22e-01],
        [1.59e-02, 8.67e-01, 1.17e-01],
        [1.55e-01, 4.22e-01, 4.22e-01]])


In [ ]:
class  CokKafaliDikkat(nn.Module):
  def __init__(self,d_model,n_kafa): # pytorch tanımlamanız gereken bir abi
    super().__init__()
    assert d_model % n_kafa == 0
    self.n_kafa = n_kafa
    self.d_k =d_model //n_kafa

    self.W_q =nn.Linear(d_model,d_model)
    self.W_k =nn.Linear(d_model,d_model)
    self.W_v =nn.Linear(d_model,d_model)
    self.W_o =nn.Linear(d_model,d_model)

    self.son_agirliklar = None

  def kafalara_bol (self,t):

    B,L,_=t.shape

    return t.view(B,L,self.n_kafa,self.d_k).transpose(1,2)

  def forward(self,x,maske=None):
    B,L,d_model =x.shape
    Q =self.kafalara_bol(self.W_q(x))
    K=self.kafalara_bol(self.W_k(x))
    V=self.kafalara_bol(self.W_v(x))

    cikti,agirliklar = olcekli_dikkat(Q,K,V,maske)

    self.son_agirliklar = agirliklar.detach()

    cikti = cikti.transpose(1,2).reshape(B,L,d_model)

    return self.W_o(cikti)

In [ ]:
deneme = torch.randn(2,5,64)
mha = CokKafaliDikkat(d_model=64,n_kafa=8)
cikti = mha(deneme)
print(cikti.shape)
print(mha.son_agirliklar.shape)

torch.Size([2, 5, 64])
torch.Size([2, 8, 5, 5])


In [ ]:
class KonumKodlamasi(nn.Module):
  def __init__(self,d_model,max_uzunluk=100):
    super().__init__()
    pe = torch.zeros(max_uzunluk,d_model)
    konum = torch.arange(max_uzunluk).unsqueeze(1).float()
    frekans = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000)/d_model))
    pe[:,0::2] = torch.sin(konum*frekans)
    pe[:,1::2] = torch.cos(konum*frekans)
    self.register_buffer("pe",pe.unsqueeze(0))
  def forward(self,x):
    return x + self.pe[:,:x.size(1)]

In [ ]:
class  IleriBeslemeli(nn.Module):
  def __init__(self,d_model,d_ff):
    super().__init__()
    self.ag =nn.Sequential(
        #boyut genişletmek için
        nn.Linear(d_model,d_ff),
        # doğrusal olmayan değerleri (negatifleri 0 değeri verme)
        nn.ReLU(),
        # model boyutuna çeirme
        nn.Linear(d_ff,d_model)
    )
  def forward(self,x):
    #her konuma ayrı ayru uygulanır
    return self.ag(x)

In [ ]:
class KodlayıcıBlok(nn.Module):
  def __init__(self,d_model,n_kafa,d_ff,dropout=0.1):
    super().__init__()
    self.dikkat = CokKafaliDikkat(d_model,n_kafa)
    self.ffn =IleriBeslemeli(d_model,d_ff)
    self.norm1 =nn.LayerNorm(d_model)
    self.norm2 =nn.LayerNorm(d_model)
    self.dropout =nn.Dropout(dropout)

  def forward(self,x,maske=None):
    x= self.norm1(x+self.dropout(self.dikkat(x,maske)))
    x= self.norm2(x+self.dropout(self.ffn(x)))

    return x


In [ ]:
class MiniTransformer(nn.Module):
  def __init__(self,sozluk_boyutu, d_model=64,n_katman=2,n_kafa=4,d_ff=128,max_uzunluk=50):
    super().__init__()
    self.gomme = nn.Embedding(sozluk_boyutu,d_model)
    self.konum_kodlamasi = KonumKodlamasi(d_model,max_uzunluk)
    self.bloklar = nn.ModuleList([
        KodlayıcıBlok(d_model,n_kafa,d_ff) for _ in range(n_katman)
    ])

    self.cikis =nn.Linear(d_model,sozluk_boyutu)

  def forward(self,x):
    h =self.gomme(x)
    h=self.konum_kodlamasi(h)
    for blok in self.bloklar:
      h=blok(h)
    return self.cikis(h)

In [ ]:
SOZLUK= 10
model = MiniTransformer(sozluk_boyutu=SOZLUK)
toplam =sum(p.numel() for p in model.parameters())
print(f"Modelimiz {toplam:,} parametreye sahiptir")


Modelimiz 68,234 parametreye sahiptir


In [ ]:
uzunluk  = 8
def veri_uret(adet):
  x= torch.randint(0,SOZLUK,(adet,uzunluk))
  y= torch.sort(x,dim=1).values
  return x,y


ornek_x ,ornek_y =veri_uret(1)
print("Girdi:",ornek_x[0].tolist())
print("Hedef:",ornek_y[0].tolist())

Girdi: [8, 4, 0, 9, 8, 2, 6, 3]
Hedef: [0, 2, 3, 4, 6, 8, 8, 9]


In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
kayip_fn = nn.CrossEntropyLoss()
kayiplar = []

model.train()

for adim in range(1,2001):
  x,y= veri_uret(64)
  logits = model(x)
  kayip=kayip_fn(logits.reshape(-1,SOZLUK),y.reshape(-1))
  optimizer.zero_grad()
  kayip.backward()
  optimizer.step()
  kayiplar.append(kayip.item())

  if adim % 200 ==0:
    print(f"Adım{adim:4d}| Kayıp:{kayip.item():.4f}")

Adım 200| Kayıp:0.0850
Adım 400| Kayıp:0.0226
Adım 600| Kayıp:0.0176
Adım 800| Kayıp:0.0123
Adım1000| Kayıp:0.0023
Adım1200| Kayıp:0.0096
Adım1400| Kayıp:0.0040
Adım1600| Kayıp:0.0015
Adım1800| Kayıp:0.0028
Adım2000| Kayıp:0.0021


In [ ]:
model.eval()
with torch.no_grad():
  x_test,y_test = veri_uret(100000)
  tahmin = model(x_test).argmax(dim=-1)

rakam_dogruluk = (tahmin==y_test).float().mean().item()
dizi_dogruluk = (tahmin==y_test).all(dim=1).float().mean().item()
print(f"Rakam Doğruluk:{rakam_dogruluk*100:.2f}%")
print(f"Dizi Doğruluk:{dizi_dogruluk*100:.2f}%")

for i in range (2):
  print("Girdi:",x_test[i].tolist())
  print("Tahmin:",tahmin[i].tolist())
  print("Hedef:",y_test[i].tolist())



Rakam Doğruluk:99.99%
Dizi Doğruluk:99.91%
Girdi: [4, 4, 4, 9, 3, 8, 3, 7]
Tahmin: [3, 3, 4, 4, 4, 7, 8, 9]
Hedef: [3, 3, 4, 4, 4, 7, 8, 9]
Girdi: [5, 5, 7, 2, 9, 8, 2, 4]
Tahmin: [2, 2, 4, 5, 5, 7, 8, 9]
Hedef: [2, 2, 4, 5, 5, 7, 8, 9]
